# 01 - Data Reading
Loads the raw Madrid and Amsterdam parquet files, builds the per-location summary (`madrid_px`, `amsterdam_px`) with the age classes, and checks for duplicates.

**Reads:** `data/madrid_train.parquet`, `data/amsterdam_data.parquet`
**Writes:** `processed/madrid_px.parquet`, `processed/amsterdam_px.parquet`
**Next:** `02_cleaning.ipynb`

## 1. Setup

In [1]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "data"          # path for data
OUT_DIR = "processed"      # hand-off files for the next notebooks
os.makedirs(OUT_DIR, exist_ok=True)
print("data dir:", os.path.abspath(DATA_DIR))
print("out dir :", os.path.abspath(OUT_DIR))

data dir: /home/participant-d4/Building_Age_Classification/data
out dir : /home/participant-d4/Building_Age_Classification/processed


## 2. Load the raw files

In [2]:
madrid = pd.read_parquet(f"{DATA_DIR}/madrid_train.parquet")
print("madrid   ", madrid.shape)

madrid    (3172765, 48)


In [3]:
amsterdam = pd.read_parquet(f"{DATA_DIR}/amsterdam_data.parquet")
print("amsterdam", amsterdam.shape)

amsterdam (1062877, 48)


## 3. First look at the raw tables

In [4]:
print("Madrid columns:", list(madrid.columns))
madrid.head()

Madrid columns: ['pixel_id', 'city', 'year', 'px_key', 'py_key', 'pixel_x', 'pixel_y', 'row', 'col', 'coverage', 'usable_label', 'weighted_mean_year', 'Blue_1', 'Green_1', 'Red_1', 'NIR_1', 'SWIR1_1', 'SWIR2_1', 'QA_PIXEL_1', 'QA_RADSAT_1', 'doy_1', 'scene_id_1', 'dataset_1', 'qa_valid_1', 'Blue_2', 'Green_2', 'Red_2', 'NIR_2', 'SWIR1_2', 'SWIR2_2', 'QA_PIXEL_2', 'QA_RADSAT_2', 'doy_2', 'scene_id_2', 'dataset_2', 'qa_valid_2', 'Blue_3', 'Green_3', 'Red_3', 'NIR_3', 'SWIR1_3', 'SWIR2_3', 'QA_PIXEL_3', 'QA_RADSAT_3', 'doy_3', 'scene_id_3', 'dataset_3', 'qa_valid_3']


,pixel_id,city,year,px_key,py_key,pixel_x,pixel_y,row,col,coverage,...,Red_3,NIR_3,SWIR1_3,SWIR2_3,QA_PIXEL_3,QA_RADSAT_3,doy_3,scene_id_3,dataset_3,qa_valid_3
0,Madrid_14194_149671_1984,Madrid,1984,14194,149671,425820.0,4490130.0,5,20,0.15,...,11603.330078,16555.982422,18056.576172,13991.788086,5440,0,247,LT05_L2SP_201032_19840903_20201008_02_T2,landsat_tm_c2_l2,True
1,Madrid_14196_149670_1984,Madrid,1984,14196,149670,425880.0,4490100.0,6,22,0.17,...,12906.125000,17859.429688,19165.214844,14418.933594,5440,0,247,LT05_L2SP_201032_19840903_20201008_02_T2,landsat_tm_c2_l2,True
2,Madrid_14259_149645_1984,Madrid,1984,14259,149645,427770.0,4489350.0,31,85,0.49,...,15025.377930,19446.933594,23392.283203,17264.550781,5440,0,247,LT05_L2SP_201032_19840903_20201008_02_T2,landsat_tm_c2_l2,True
3,Madrid_14268_149501_1984,Madrid,1984,14268,149501,428040.0,4485030.0,175,94,0.28,...,11716.317383,16982.070312,17746.361328,13844.357422,5440,0,247,LT05_L2SP_201032_19840903_20201008_02_T2,landsat_tm_c2_l2,True
4,Madrid_14302_149354_1984,Madrid,1984,14302,149354,429060.0,4480620.0,322,128,0.28,...,11596.452148,16977.035156,15927.785156,12702.273438,5440,0,247,LT05_L2SP_201032_19840903_20201008_02_T2,landsat_tm_c2_l2,True


In [5]:
print("Amsterdam columns:", list(amsterdam.columns))
amsterdam.head()

Amsterdam columns: ['pixel_id', 'city', 'year', 'px_key', 'py_key', 'pixel_x', 'pixel_y', 'row', 'col', 'coverage', 'usable_label', 'weighted_mean_year', 'Blue_1', 'Green_1', 'Red_1', 'NIR_1', 'SWIR1_1', 'SWIR2_1', 'QA_PIXEL_1', 'QA_RADSAT_1', 'doy_1', 'scene_id_1', 'dataset_1', 'qa_valid_1', 'Blue_2', 'Green_2', 'Red_2', 'NIR_2', 'SWIR1_2', 'SWIR2_2', 'QA_PIXEL_2', 'QA_RADSAT_2', 'doy_2', 'scene_id_2', 'dataset_2', 'qa_valid_2', 'Blue_3', 'Green_3', 'Red_3', 'NIR_3', 'SWIR1_3', 'SWIR2_3', 'QA_PIXEL_3', 'QA_RADSAT_3', 'doy_3', 'scene_id_3', 'dataset_3', 'qa_valid_3']


,pixel_id,city,year,px_key,py_key,pixel_x,pixel_y,row,col,coverage,...,Red_3,NIR_3,SWIR1_3,SWIR2_3,QA_PIXEL_3,QA_RADSAT_3,doy_3,scene_id_3,dataset_3,qa_valid_3
0,Amsterdam_20600_193589_1984,Amsterdam,1984,20600,193589,618000.0,5807670.0,107,35,0.23,...,21618.935547,22834.484375,14306.528320,12677.02832,13600,0,322,LT05_L2SP_198024_19841117_20200918_02_T2,landsat_tm_c2_l2,False
1,Amsterdam_20600_193590_1984,Amsterdam,1984,20600,193590,618000.0,5807700.0,106,35,0.19,...,22280.158203,22834.484375,14306.528320,12677.02832,13600,0,322,LT05_L2SP_198024_19841117_20200918_02_T2,landsat_tm_c2_l2,False
2,Amsterdam_20601_193591_1984,Amsterdam,1984,20601,193591,618030.0,5807730.0,105,36,0.98,...,22608.314453,23219.273438,14843.556641,13456.59375,13600,0,322,LT05_L2SP_198024_19841117_20200918_02_T2,landsat_tm_c2_l2,False
3,Amsterdam_20601_193593_1984,Amsterdam,1984,20601,193593,618030.0,5807790.0,103,36,0.20,...,22608.314453,23605.068359,14575.042969,13456.59375,13600,0,322,LT05_L2SP_198024_19841117_20200918_02_T2,landsat_tm_c2_l2,False
4,Amsterdam_20602_193588_1984,Amsterdam,1984,20602,193588,618060.0,5807640.0,108,37,0.55,...,21618.935547,23219.273438,14306.528320,12677.02832,13600,0,322,LT05_L2SP_198024_19841117_20200918_02_T2,landsat_tm_c2_l2,False


## 4. Class boundaries and the location summary function

In [6]:
# City-specific class boundaries. The 1/2 split differs; 1984 and 2004 don't.
BOUNDS = {"madrid": [-float("inf"), 1960, 1984, 2004, float("inf")],
          "amsterdam": [-float("inf"), 1945, 1984, 2004, float("inf")]}

def summarise(df, city):
    key = ["px_key", "py_key"]                      # this identifies a location

    px = df.groupby(key).agg(
        weighted_mean_year=("weighted_mean_year", "first"),
        coverage=("coverage", "first"),
        n_years=("year", "nunique"),
    )
    px["age_class"] = pd.cut(px["weighted_mean_year"],
                             bins=BOUNDS[city], labels=[1, 2, 3, 4]).astype(int)

    # Only check observation slots that are actually flagged valid
    bad = 0
    for n in (1, 2, 3):
        qa = f"qa_valid_{n}"
        if qa not in df.columns:
            continue
        bands = [f"{b}_{n}" for b in ("Blue","Green","Red","NIR","SWIR1","SWIR2")
                 if f"{b}_{n}" in df.columns]
        sub = df.loc[df[qa].fillna(False).astype(bool), bands]
        bad += ((sub <= 0) | (sub >= 65535)).any(axis=1).sum()

    print(f"\n=== {city.upper()} ===")
    print(f"rows {len(df):,}  locations {len(px):,}  "
          f"median years/location {px.n_years.median():.0f}")
    print(px.age_class.value_counts().sort_index()
            .to_frame("pixels").assign(pct=lambda d: (100*d.pixels/len(px)).round(1)))
    print(f"corrupt rows: {bad}   coverage mean: {px.coverage.mean():.3f}")
    return px
print("OK - defined: summarise")

OK - defined: summarise


## 5. Summarise each city

In [7]:
madrid_px = summarise(madrid, "madrid")
print("OK - ready")


=== MADRID ===
rows 3,172,765  locations 76,263  median years/location 42
           pixels   pct
age_class              
1           13794  18.1
2           27152  35.6
3           18672  24.5
4           16645  21.8
corrupt rows: 5031   coverage mean: 0.505
OK - ready


In [8]:
amsterdam_px = summarise(amsterdam, "amsterdam")
print("OK - ready")


=== AMSTERDAM ===
rows 1,062,877  locations 25,992  median years/location 41
           pixels   pct
age_class              
1            7568  29.1
2            8802  33.9
3            6745  26.0
4            2877  11.1
corrupt rows: 372   coverage mean: 0.421
OK - ready


## 6. Check for duplicates

In [9]:
# Check for duplicates (same location + year should appear once)
for df, name in [(madrid, "madrid"), (amsterdam, "amsterdam")]:
    print(name, df.duplicated(subset=["px_key", "py_key", "year"]).sum())

madrid 0
amsterdam 0


## 7. Save for the next notebook

In [10]:
madrid_px.to_parquet(f"{OUT_DIR}/madrid_px.parquet")
amsterdam_px.to_parquet(f"{OUT_DIR}/amsterdam_px.parquet")
print("saved:", sorted(f for f in os.listdir(OUT_DIR) if f.endswith("_px.parquet")))

saved: ['amsterdam_px.parquet', 'madrid_px.parquet']
